Name : Vaishnavi Khopkar

Roll No : 23102A0074

Branch : CMPN-A

# Laboratory Assignment 8:



## 1. Dataset
* Yellow Taxi Trip Records (Parquet) from the official TLC site: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
* Taxi Zone Lookup Table (CSV) from the same page.
* This notebook uses **Jan–Mar 2024 (~9M trips)**. Change `MONTHS` below to use more or fewer.

 Setup

In [ ]:
# Use prebuilt Linux binaries (much faster than compiling, esp. for arrow)
os_cn <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = paste0("https://packagemanager.posit.co/cran/__linux__/", os_cn, "/latest")),
        HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                                paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))

pkgs <- c("data.table","ggplot2","foreach","doParallel","microbenchmark","purrr","lubridate","arrow","scales")
new  <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(new)) install.packages(new)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘iterators’, ‘assertthat’




In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(foreach); library(doParallel)
  library(microbenchmark); library(purrr); library(lubridate); library(scales)
})
cat("Cores available:", parallel::detectCores(), "\n")
cat("RAM check:\n"); system("free -h")

Cores available: 2 
RAM check:


## 4. Task 1: Data Acquisition and Preparation
### 4.1 Import
Only the required columns are read from the Parquet files (column pruning saves memory).

In [ ]:
MONTHS <- sprintf("2024-%02d", 1:3)          # <-- change to use more/less data
base   <- "https://d37ci6vzurychx.cloudfront.net/trip-data/"
dir.create("data", showWarnings = FALSE)

files <- character()
for (m in MONTHS) {
  f <- file.path("data", sprintf("yellow_tripdata_%s.parquet", m))
  if (!file.exists(f)) download.file(sprintf("%syellow_tripdata_%s.parquet", base, m), f, mode = "wb", quiet = TRUE)
  files <- c(files, f)
}

keep <- c("tpep_pickup_datetime","tpep_dropoff_datetime","passenger_count","trip_distance",
          "PULocationID","DOLocationID","payment_type","fare_amount","tip_amount","tolls_amount","total_amount")

t_load <- system.time({
  dt <- rbindlist(lapply(files, function(f) as.data.table(arrow::read_parquet(f, col_select = keep))))
})
print(t_load)
cat("Rows:", format(nrow(dt), big.mark=","), " Cols:", ncol(dt), "\n")

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(keep)

  # Now:
  data %>% select(all_of(keep))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


   user  system elapsed 
  3.843   1.451   4.496 
Rows: 9,554,778  Cols: 11 


### 4.2 Inspect structure, dimensions, data types, summary

In [ ]:
dim(dt)
str(dt)
print(format(object.size(dt), units = "MB"))
summary(dt)
colSums(is.na(dt))

[1] 9554778      11

Classes ‘data.table’ and 'data.frame':	9554778 obs. of  11 variables:
 $ tpep_pickup_datetime : POSIXct, format: "2024-01-01 00:57:55" "2024-01-01 00:03:00" ...
 $ tpep_dropoff_datetime: POSIXct, format: "2024-01-01 01:17:43" "2024-01-01 00:09:36" ...
 $ passenger_count      : int  1 1 1 1 1 1 2 0 1 1 ...
 $ trip_distance        : num  1.72 1.8 4.7 1.4 0.8 ...
 $ PULocationID         : int  186 140 236 79 211 148 138 246 161 113 ...
 $ DOLocationID         : int  79 236 79 211 148 141 181 231 261 113 ...
 $ payment_type         : int  2 1 1 1 1 1 1 2 2 2 ...
 $ fare_amount          : num  17.7 10 23.3 10 7.9 29.6 45.7 25.4 31 3 ...
 $ tip_amount           : num  0 3.75 3 2 3.2 6.9 10 0 0 0 ...
 $ tolls_amount         : num  0 0 0 0 0 0 0 0 0 0 ...
 $ total_amount         : num  22.7 18.8 31.3 17 16.1 ...
 - attr(*, ".internal.selfref")=<pointer: 0x5c7ff85503f0> 
[1] "656.1 Mb"


 tpep_pickup_datetime          tpep_dropoff_datetime         passenger_count 
 Min.   :2002-12-31 22:17:10   Min.   :2002-12-31 22:42:24   Min.   :0.000   
 1st Qu.:2024-01-26 09:55:41   1st Qu.:2024-01-26 10:12:00   1st Qu.:1.000   
 Median :2024-02-18 15:38:32   Median :2024-02-18 15:55:02   Median :1.000   
 Mean   :2024-02-17 17:20:27   Mean   :2024-02-17 17:36:34   Mean   :1.334   
 3rd Qu.:2024-03-11 11:59:11   3rd Qu.:2024-03-11 12:16:17   3rd Qu.:1.000   
 Max.   :2024-04-01 00:34:55   Max.   :2024-04-02 18:08:46   Max.   :9.000   
                                                             NAs    :751962  
 trip_distance        PULocationID    DOLocationID    payment_type  
 Min.   :0.000e+00   Min.   :  1.0   Min.   :  1.0   Min.   :0.000  
 1st Qu.:1.000e+00   1st Qu.:132.0   1st Qu.:114.0   1st Qu.:1.000  
 Median :1.700e+00   Median :162.0   Median :162.0   Median :1.000  
 Mean   :4.042e+00   Mean   :165.2   Mean   :164.2   Mean   :1.121  
 3rd Qu.:3.190e+00   3rd Qu.:23

tpep_pickup_datetime tpep_dropoff_datetime       passenger_count 
                    0                     0                751962 
        trip_distance          PULocationID          DOLocationID 
                    0                     0                     0 
         payment_type           fare_amount            tip_amount 
                    0                     0                     0 
         tolls_amount          total_amount 
                    0                     0

### 4.3 Handle missing, duplicate, inconsistent and invalid records

In [ ]:
n0 <- nrow(dt)
log_steps <- data.table(step = "Raw", rows = n0)
log_it <- function(s) log_steps <<- rbind(log_steps, data.table(step = s, rows = nrow(dt)))

# 1. missing values in key columns
dt <- na.omit(dt, cols = c("passenger_count","payment_type"))
log_it("Remove NA passenger_count/payment_type")

# 2. exact duplicates
dt <- unique(dt)
log_it("Remove duplicates")

# 3. invalid / inconsistent values
dt[, duration_min := as.numeric(difftime(tpep_dropoff_datetime, tpep_pickup_datetime, units = "mins"))]
start <- as.POSIXct(paste0(MONTHS[1], "-01"), tz = "UTC")
end   <- seq(as.Date(paste0(tail(MONTHS,1), "-01")), by = "month", length.out = 2)[2]
dt <- dt[tpep_pickup_datetime >= start & tpep_pickup_datetime < as.POSIXct(end, tz="UTC")]
log_it("Pickup date outside selected months")
dt <- dt[duration_min > 1 & duration_min < 360]
log_it("Duration not in (1, 360) min")
dt <- dt[trip_distance > 0 & trip_distance < 100]
log_it("Distance not in (0, 100) miles")
dt <- dt[fare_amount > 0 & fare_amount < 500 & total_amount > 0 & tip_amount >= 0]
log_it("Invalid fare/total/tip")
dt <- dt[passenger_count >= 1 & passenger_count <= 6]
log_it("Passenger count not in 1-6")
dt[, speed_mph := trip_distance / (duration_min / 60)]
dt <- dt[speed_mph < 80]
log_it("Speed > 80 mph")

log_steps[, removed := shift(rows) - rows]
print(log_steps)
cat(sprintf("Kept %.2f%% of raw records\n", 100*nrow(dt)/n0))

                                     step    rows removed
                                   <char>   <int>   <int>
1:                                    Raw 9554778      NA
2: Remove NA passenger_count/payment_type 8802816  751962
3:                      Remove duplicates 8802815       1
4:    Pickup date outside selected months 8802794      21
5:           Duration not in (1, 360) min 8690685  112109
6:         Distance not in (0, 100) miles 8648645   42040
7:                 Invalid fare/total/tip 8549409   99236
8:             Passenger count not in 1-6 8447626  101783
9:                         Speed > 80 mph 8447421     205
Kept 88.41% of raw records


### 4.4 Convert types, extract temporal attributes, drop unneeded columns

In [ ]:
pay_lbl <- c("0"="Flex fare","1"="Credit card","2"="Cash","3"="No charge","4"="Dispute","5"="Unknown","6"="Voided")
dt[, `:=`(
  hour       = data.table::hour(tpep_pickup_datetime),
  day        = data.table::mday(tpep_pickup_datetime),
  wday_num   = data.table::wday(tpep_pickup_datetime),      # 1 = Sunday
  month      = data.table::month(tpep_pickup_datetime),
  PULocationID = as.integer(PULocationID),
  DOLocationID = as.integer(DOLocationID),
  passenger_count = as.integer(passenger_count),
  payment    = factor(pay_lbl[as.character(payment_type)])
)]
dt[, wday := factor(wday_num, levels = 1:7, labels = c("Sun","Mon","Tue","Wed","Thu","Fri","Sat"))]
dt[, period := cut(hour, c(-1,5,11,16,20,23), labels = c("Night (0-5)","Morning (6-11)","Afternoon (12-16)","Evening (17-20)","Late night (21-23)"))]

before <- object.size(dt)
dt[, c("tpep_dropoff_datetime","payment_type","wday_num") := NULL]   # drop unneeded
invisible(gc())
cat("Memory before:", format(before, units="MB"), " after:", format(object.size(dt), units="MB"), "\n")
str(dt)

Memory before: 934.5 Mb  after: 805.6 Mb 
Classes ‘data.table’ and 'data.frame':	8447421 obs. of  17 variables:
 $ tpep_pickup_datetime: POSIXct, format: "2024-01-01 00:57:55" "2024-01-01 00:03:00" ...
 $ passenger_count     : int  1 1 1 1 1 1 2 1 1 2 ...
 $ trip_distance       : num  1.72 1.8 4.7 1.4 0.8 ...
 $ PULocationID        : int  186 140 236 79 211 148 138 161 113 107 ...
 $ DOLocationID        : int  79 236 79 211 148 141 181 261 113 137 ...
 $ fare_amount         : num  17.7 10 23.3 10 7.9 29.6 45.7 31 3 7.9 ...
 $ tip_amount          : num  0 3.75 3 2 3.2 6.9 10 0 0 0 ...
 $ tolls_amount        : num  0 0 0 0 0 0 0 0 0 0 ...
 $ total_amount        : num  22.7 18.8 31.3 17 16.1 ...
 $ duration_min        : num  19.8 6.6 17.9 8.3 6.1 ...
 $ speed_mph           : num  5.21 16.36 15.74 10.12 7.87 ...
 $ hour                : int  0 0 0 0 0 0 0 0 0 0 ...
 $ day                 : int  1 1 1 1 1 1 1 1 1 1 ...
 $ month               : int  1 1 1 1 1 1 1 1 1 1 ...
 $ payment        

### 4.5 Import Taxi Zone Lookup and join (pickup and drop-off)

In [ ]:
zones <- fread("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
print(head(zones))

setkey(zones, LocationID)
t_join <- system.time({
  dt[zones, on = c(PULocationID = "LocationID"), `:=`(PU_Borough = i.Borough, PU_Zone = i.Zone)]
  dt[zones, on = c(DOLocationID = "LocationID"), `:=`(DO_Borough = i.Borough, DO_Zone = i.Zone)]
})
print(t_join)
dt[is.na(PU_Zone), PU_Zone := "Unknown"][is.na(DO_Zone), DO_Zone := "Unknown"]
dt[is.na(PU_Borough), PU_Borough := "Unknown"][is.na(DO_Borough), DO_Borough := "Unknown"]
head(dt)

   LocationID       Borough                    Zone service_zone
        <int>        <char>                  <char>       <char>
1:          1           EWR          Newark Airport          EWR
2:          2        Queens             Jamaica Bay    Boro Zone
3:          3         Bronx Allerton/Pelham Gardens    Boro Zone
4:          4     Manhattan           Alphabet City  Yellow Zone
5:          5 Staten Island           Arden Heights    Boro Zone
6:          6 Staten Island Arrochar/Fort Wadsworth    Boro Zone
   user  system elapsed 
  2.623   0.418   3.078 


tpep_pickup_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,tip_amount,tolls_amount,total_amount,duration_min,⋯,hour,day,month,payment,wday,period,PU_Borough,PU_Zone,DO_Borough,DO_Zone
<dttm>,<int>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<int>,<int>,<int>,<fct>,<fct>,<fct>,<chr>,<chr>,<chr>,<chr>
2024-01-01 00:57:55,1,1.72,186,79,17.7,0.00,0,22.70,19.80000,⋯,0,1,1,Cash,Mon,Night (0-5),Manhattan,Penn Station/Madison Sq West,Manhattan,East Village
2024-01-01 00:03:00,1,1.80,140,236,10.0,3.75,0,18.75,6.60000,⋯,0,1,1,Credit card,Mon,Night (0-5),Manhattan,Lenox Hill East,Manhattan,Upper East Side North
2024-01-01 00:17:06,1,4.70,236,79,23.3,3.00,0,31.30,17.91667,⋯,0,1,1,Credit card,Mon,Night (0-5),Manhattan,Upper East Side North,Manhattan,East Village
2024-01-01 00:36:38,1,1.40,79,211,10.0,2.00,0,17.00,8.30000,⋯,0,1,1,Credit card,Mon,Night (0-5),Manhattan,East Village,Manhattan,SoHo
2024-01-01 00:46:51,1,0.80,211,148,7.9,3.20,0,16.10,6.10000,⋯,0,1,1,Credit card,Mon,Night (0-5),Manhattan,SoHo,Manhattan,Lower East Side
2024-01-01 00:54:08,1,4.70,148,141,29.6,6.90,0,41.50,32.38333,⋯,0,1,1,Credit card,Mon,Night (0-5),Manhattan,Lower East Side,Manhattan,Lenox Hill West


## 5. Task 2: Transportation Data Analytics (data.table)

In [ ]:
# 1. Trips by hour
trips_hour <- dt[, .(trips = .N), by = hour][order(hour)]
print(trips_hour)
# 2. Day of week
trips_wday <- dt[, .(trips = .N, avg_fare = mean(fare_amount), revenue = sum(total_amount)), by = wday][order(wday)]
print(trips_wday)
# 3. Monthly
trips_month <- dt[, .(trips = .N, avg_fare = mean(fare_amount), revenue = sum(total_amount)), by = month][order(month)]
print(trips_month)

     hour  trips
    <int>  <int>
 1:     0 221078
 2:     1 147457
 3:     2  97985
 4:     3  65498
 5:     4  40249
 6:     5  45755
 7:     6 108508
 8:     7 226837
 9:     8 321475
10:     9 367863
11:    10 397731
12:    11 428804
13:    12 467733
14:    13 481475
15:    14 517018
16:    15 531997
17:    16 542481
18:    17 588113
19:    18 609239
20:    19 535294
21:    20 479767
22:    21 477389
23:    22 427010
24:    23 320665
     hour  trips
    <int>  <int>
     wday   trips avg_fare  revenue
   <fctr>   <int>    <num>    <num>
1:    Sun 1049105 19.34713 29595518
2:    Mon 1029384 19.64266 29947369
3:    Tue 1167928 18.53973 32420114
4:    Wed 1277651 18.52868 35502464
5:    Thu 1363149 18.81526 38394113
6:    Fri 1277835 18.48267 35395386
7:    Sat 1282369 17.36555 32670155
   month   trips avg_fare  revenue
   <int>   <int>    <num>    <num>
1:     1 2713394 18.38915 74244060
2:     2 2709713 18.33664 74090268
3:     3 3024314 19.10981 85590791


In [ ]:
# 4. Average fare and revenue by time period
fare_hour   <- dt[, .(avg_fare = mean(fare_amount), revenue = sum(total_amount)), by = hour][order(hour)]
fare_period <- dt[, .(trips = .N, avg_fare = mean(fare_amount), revenue = sum(total_amount)), by = period][order(period)]
print(fare_period)

# 5. Most frequent routes
routes <- dt[, .(trips = .N, revenue = sum(total_amount), avg_fare = mean(fare_amount), avg_dist = mean(trip_distance)),
             by = .(PU_Zone, DO_Zone)]
top_routes_freq <- routes[order(-trips)][1:15]
print(top_routes_freq)

# 6. Highest-revenue routes
top_routes_rev <- routes[order(-revenue)][1:15]
print(top_routes_rev)

# Top pickup / drop-off zones
top_pu <- dt[, .(trips = .N), by = PU_Zone][order(-trips)][1:15]
top_do <- dt[, .(trips = .N), by = DO_Zone][order(-trips)][1:15]

               period   trips avg_fare  revenue
               <fctr>   <int>    <num>    <num>
1:        Night (0-5)  618022 18.85487 17154115
2:     Morning (6-11) 1851218 18.27942 48693858
3:  Afternoon (12-16) 2540704 19.39224 71771257
4:    Evening (17-20) 2212413 17.78148 61455407
5: Late night (21-23) 1225064 19.00006 34850483
                         PU_Zone                   DO_Zone trips  revenue
                          <char>                    <char> <int>    <num>
 1:        Upper East Side South     Upper East Side North 61418 958743.4
 2:        Upper East Side North     Upper East Side South 53927 864287.4
 3:        Upper East Side North     Upper East Side North 42599 556841.7
 4:        Upper East Side South     Upper East Side South 39796 545236.1
 5:               Midtown Center     Upper East Side South 29789 506686.0
 6:          Lincoln Square East     Upper West Side South 25473 381839.6
 7:        Upper East Side South            Midtown Center 25377 424474.

In [ ]:
# 7. Trip distance vs fare
cat("Pearson correlation (distance, fare):", cor(dt$trip_distance, dt$fare_amount), "\n")
fit <- lm(fare_amount ~ trip_distance, data = dt[sample(.N, min(.N, 1e6))])
print(summary(fit)$coefficients); cat("R-squared:", summary(fit)$r.squared, "\n")
dist_bins <- dt[, .(trips=.N, avg_fare = mean(fare_amount), avg_per_mile = mean(fare_amount/trip_distance)),
                by = .(dist_bin = cut(trip_distance, c(0,1,2,5,10,20,100)))][order(dist_bin)]
print(dist_bins)

# 8. Payment methods across boroughs
pay_borough <- dt[, .(trips = .N, avg_tip = mean(tip_amount)), by = .(PU_Borough, payment)]
pay_borough[, share := trips / sum(trips), by = PU_Borough]
print(pay_borough[order(PU_Borough, -trips)])

Pearson correlation (distance, fare): 0.9546767 
              Estimate  Std. Error  t value Pr(>|t|)
(Intercept)   6.345889 0.006339532 1001.003        0
trip_distance 3.714041 0.001152086 3223.752        0
R-squared: 0.9122237 
   dist_bin   trips  avg_fare avg_per_mile
     <fctr>   <int>     <num>        <num>
1:    (0,1] 2064327  7.580306    12.642536
2:    (1,2] 2837431 11.670061     8.084600
3:    (2,5] 2196685 18.918434     6.450227
4:   (5,10]  674966 34.553280     4.788520
5:  (10,20]  588870 61.292043     4.148112
6: (20,100]   85142 89.776077     3.679519
       PU_Borough     payment   trips      avg_tip       share
           <char>      <fctr>   <int>        <num>       <num>
 1:         Bronx Credit card   14065 2.402979e-01 0.915392125
 2:         Bronx        Cash    1181 0.000000e+00 0.076863000
 3:         Bronx     Dispute      69 0.000000e+00 0.004490726
 4:         Bronx   No charge      50 0.000000e+00 0.003254149
 5:      Brooklyn Credit card   50768 1.664713e+

In [ ]:
# 9. Additional patterns
# (a) Airport trips
airport <- dt[grepl("Airport|JFK|LaGuardia|Newark", PU_Zone), .(trips=.N, avg_fare=mean(fare_amount), avg_total=mean(total_amount)), by = PU_Zone][order(-trips)]
print(airport)
# (b) Tip percentage by payment type (cash tips are not recorded!)
tip_pay <- dt[total_amount > 0, .(avg_tip_pct = 100*mean(tip_amount/fare_amount)), by = payment][order(-avg_tip_pct)]
print(tip_pay)
# (c) Average speed by hour (congestion)
speed_hour <- dt[, .(avg_speed = mean(speed_mph)), by = hour][order(hour)]
print(speed_hour)
# (d) Intra-zone trips and passenger-count distribution
cat("Share of trips within same zone:", dt[PULocationID == DOLocationID, .N]/nrow(dt), "\n")
print(dt[, .(trips = .N), by = passenger_count][order(passenger_count)])

             PU_Zone  trips avg_fare avg_total
              <char>  <int>    <num>     <num>
1:       JFK Airport 400232 63.66373  82.14281
2: LaGuardia Airport 274039 42.73490  67.07657
3:    Newark Airport     69 77.70217  96.27116
       payment  avg_tip_pct
        <fctr>        <num>
1: Credit card 25.340142689
2:     Dispute  0.101110462
3:   No charge  0.031427036
4:        Cash  0.001351631
     hour avg_speed
    <int>     <num>
 1:     0 14.323454
 2:     1 13.633106
 3:     2 13.730889
 4:     3 14.901695
 5:     4 17.998038
 6:     5 19.684457
 7:     6 16.130458
 8:     7 12.902080
 9:     8 10.943195
10:     9 10.780252
11:    10 10.558415
12:    11 10.005384
13:    12  9.986389
14:    13 10.079966
15:    14  9.864516
16:    15  9.638749
17:    16  9.808457
18:    17  9.631141
19:    18 10.130755
20:    19 11.368130
21:    20 12.387756
22:    21 12.871752
23:    22 13.227459
24:    23 14.205097
     hour avg_speed
    <int>     <num>
Share of trips within same zone: 0.04

### Interpretation (fill in after running)
* **Hourly demand:** peaks at ___ h; lowest at ___ h because ___.
* **Day of week:** busiest day ___; weekends differ because ___.
* **Monthly:** ___ had the highest demand/revenue.
* **Routes:** the most frequent routes are mostly ___ (e.g. within Manhattan / airport links); highest-revenue routes are dominated by ___ because longer distance → higher fare.
* **Distance vs fare:** correlation ≈ ___; roughly linear (base fare + per-mile rate), outliers are flat-rate airport trips.
* **Payment:** credit card ≈ ___%, cash ≈ ___%; recorded tips are ≈ 0 for cash because cash tips are not captured.

## 6. Task 3: Functional vs Vectorized vs data.table
Two representative operations:
* **Op A (group aggregation):** average fare per hour
* **Op B (row-wise computation):** tip percentage per trip (100k rows – `apply` row-wise is very slow)

In [ ]:
set.seed(123)
S    <- dt[sample(.N, min(.N, 1e6)), .(hour, fare_amount, tip_amount)]
S_df <- as.data.frame(S)
hrs  <- 0:23

# ---- Op A ----
f_base   <- function() aggregate(fare_amount ~ hour, data = S_df, FUN = mean)                       # base R
f_tapply <- function() tapply(S$fare_amount, S$hour, mean)                                          # vectorized
f_lapply <- function() lapply(hrs, function(h) mean(S$fare_amount[S$hour == h]))                    # functional
f_sapply <- function() sapply(hrs, function(h) mean(S$fare_amount[S$hour == h]))
f_map    <- function() purrr::map_dbl(hrs, ~ mean(S$fare_amount[S$hour == .x]))                     # purrr
f_dt     <- function() S[, .(avg_fare = mean(fare_amount)), by = hour]                              # data.table

# correctness check
stopifnot(all.equal(as.numeric(f_tapply()), f_dt()[order(hour), avg_fare]),
          all.equal(as.numeric(f_sapply()), f_dt()[order(hour), avg_fare]))
cat("All Op-A implementations give identical results\n")

All Op-A implementations give identical results


In [ ]:
# ---- Op B (100k rows) ----
N  <- 1e5
SB <- S[1:N]
M  <- as.matrix(SB[, .(fare_amount, tip_amount)])

g_apply  <- function() apply(M, 1, function(r) 100 * r[2] / r[1])                                   # apply (row-wise)
g_lapply <- function() unlist(lapply(seq_len(N), function(i) 100 * SB$tip_amount[i] / SB$fare_amount[i]))
g_map2   <- function() purrr::map2_dbl(SB$tip_amount, SB$fare_amount, ~ 100 * .x / .y)              # purrr
g_vec    <- function() 100 * SB$tip_amount / SB$fare_amount                                         # vectorized
g_dt     <- function() { x <- copy(SB); x[, tip_pct := 100 * tip_amount / fare_amount]; x$tip_pct } # data.table

stopifnot(all.equal(g_apply(), g_vec()), all.equal(g_map2(), g_vec()), all.equal(g_dt(), g_vec()))
cat("All Op-B implementations give identical results\n")

All Op-B implementations give identical results


In [ ]:
# Peak memory helper (Mb allocated during the call)
mem_peak <- function(f) {
  invisible(gc(reset = TRUE)); base <- sum(gc()[, 2]); invisible(gc(reset = TRUE))
  r <- f(); mx <- sum(gc()[, 6]); mx - base
}
mem_tab <- data.table(
  Task = c(rep("Op A: avg fare by hour", 5), rep("Op B: tip % per row", 5)),
  Approach = c("Base aggregate","tapply (vectorized)","lapply","purrr::map_dbl","data.table",
               "apply (row-wise)","lapply (row-wise)","purrr::map2_dbl","Vectorized","data.table"),
  Peak_MB = c(mem_peak(f_base), mem_peak(f_tapply), mem_peak(f_lapply), mem_peak(f_map), mem_peak(f_dt),
              mem_peak(g_apply), mem_peak(g_lapply), mem_peak(g_map2), mem_peak(g_vec), mem_peak(g_dt)))
print(mem_tab)

                      Task            Approach Peak_MB
                    <char>              <char>   <num>
 1: Op A: avg fare by hour      Base aggregate   190.0
 2: Op A: avg fare by hour tapply (vectorized)    35.8
 3: Op A: avg fare by hour              lapply   194.7
 4: Op A: avg fare by hour      purrr::map_dbl   194.7
 5: Op A: avg fare by hour          data.table    28.4
 6:    Op B: tip % per row    apply (row-wise)   179.2
 7:    Op B: tip % per row   lapply (row-wise)   169.3
 8:    Op B: tip % per row     purrr::map2_dbl   142.8
 9:    Op B: tip % per row          Vectorized     0.8
10:    Op B: tip % per row          data.table     4.7


### Code complexity / readability notes
| Approach | Lines | Readability | Comment |
|---|---|---|---|
| Base `aggregate` | 1 | Good | formula interface, but slow on millions of rows |
| `lapply`/`map` | 1-2 | Moderate | scans the full vector 24 times (once per hour) |
| `apply` row-wise | 1 | Moderate | converts to matrix and calls an R function per row |
| Vectorized | 1 | Excellent | single C-level pass |
| `data.table` | 1 | Excellent | `DT[i, j, by]` syntax, multithreaded, grouped in a single pass |

## 7. Task 5 (part 1): Benchmarking with `microbenchmark`

In [ ]:
mb_tab <- function(mb, task) { s <- summary(mb, unit = "ms"); data.table(Task = task, Approach = as.character(s$expr), Median_ms = s$median, Mean_ms = s$mean, Max_ms = s$max) }

mbA <- microbenchmark(Base_aggregate = f_base(), tapply_vectorized = f_tapply(), lapply = f_lapply(),
                      purrr_map = f_map(), data.table = f_dt(), times = 10)
mbB <- microbenchmark(apply_rowwise = g_apply(), lapply_rowwise = g_lapply(), purrr_map2 = g_map2(),
                      vectorized = g_vec(), data.table = g_dt(), times = 5)
print(mbA); print(mbB)
res_A <- mb_tab(mbA, "Op A: avg fare by hour (1M rows)")
res_B <- mb_tab(mbB, "Op B: tip % per row (100k rows)")

Unit: milliseconds
              expr       min        lq      mean    median        uq       max
    Base_aggregate 369.09917 408.19981 494.72255 420.96165 621.19593 683.84379
 tapply_vectorized  55.87433  56.07927  64.40711  59.44908  70.45626  87.97220
            lapply  99.23122 101.68295 124.44342 117.38926 137.87419 185.29791
         purrr_map  98.89316  99.53085 112.72201 104.16584 129.71279 143.55544
        data.table  23.79409  24.03660  24.37650  24.21995  24.60030  25.74148
 neval
    10
    10
    10
    10
    10
Unit: microseconds
           expr        min         lq        mean     median         uq
  apply_rowwise 603720.799 628757.665 702411.5638 708445.157 740643.730
 lapply_rowwise 394641.373 396033.753 422961.9792 403893.417 425464.922
     purrr_map2 194672.057 219518.563 260711.2718 229602.707 229752.340
     vectorized    343.485    345.961    370.6068    362.057    369.980
     data.table   1751.038   2036.655   2080.2258   2086.450   2096.453
        max ne

In [ ]:
# Base R vs data.table on the FULL cleaned dataset (system.time)
dt_df <- as.data.frame(dt[, .(hour, fare_amount, total_amount)])
t_base_full <- system.time(aggregate(cbind(fare_amount, total_amount) ~ hour, data = dt_df, FUN = mean))["elapsed"]
t_dt_full   <- system.time(dt[, .(fare = mean(fare_amount), tot = mean(total_amount)), by = hour])["elapsed"]
cat(sprintf("FULL data (%s rows): Base aggregate = %.2fs | data.table = %.2fs | speedup = %.1fx\n",
            format(nrow(dt), big.mark=","), t_base_full, t_dt_full, t_base_full / t_dt_full))
rm(dt_df); invisible(gc())

# Scalability: time vs data size
sizes <- c(1e5, 5e5, 1e6, 2e6, 5e6); sizes <- sizes[sizes <= nrow(dt)]
scal <- rbindlist(lapply(sizes, function(n) {
  idx <- sample.int(nrow(dt), n); sub <- dt[idx, .(hour, fare_amount)]; sdf <- as.data.frame(sub)
  data.table(n = n,
    Base_aggregate = system.time(aggregate(fare_amount ~ hour, sdf, mean))["elapsed"],
    tapply         = system.time(tapply(sub$fare_amount, sub$hour, mean))["elapsed"],
    purrr_map      = system.time(purrr::map_dbl(hrs, ~ mean(sub$fare_amount[sub$hour == .x])))["elapsed"],
    data.table     = system.time(sub[, mean(fare_amount), by = hour])["elapsed"])
}))
print(scal)

FULL data (8,447,421 rows): Base aggregate = 7.32s | data.table = 0.31s | speedup = 23.9x
       n Base_aggregate tapply purrr_map data.table
   <num>          <num>  <num>     <num>      <num>
1: 1e+05          0.076  0.006     0.010      0.004
2: 5e+05          0.179  0.028     0.052      0.013
3: 1e+06          0.373  0.059     0.168      0.037
4: 2e+06          1.164  0.115     0.200      0.049
5: 5e+06          2.388  0.287     0.664      0.114


## 8. Task 4: Parallel Computing
**Partition:** by month (each month is an independent subset).
**Computationally intensive operation:** per month, a bootstrap (B resamples) of the fare ~ distance regression slope using `lm.fit`.

In [ ]:
month_list <- split(dt[, .(month, trip_distance, fare_amount)], by = "month", keep.by = TRUE)
names(month_list) <- paste0("M", names(month_list))

boot_month <- function(d, B = 50, n_sample = 5e5) {
  set.seed(42 + d$month[1])
  d <- d[sample.int(nrow(d), min(nrow(d), n_sample))]
  X <- cbind(1, d$trip_distance); y <- d$fare_amount; n <- length(y)
  co <- t(replicate(B, { i <- sample.int(n, n, replace = TRUE); lm.fit(X[i, , drop = FALSE], y[i])$coefficients }))
  data.table(month = d$month[1], slope_mean = mean(co[,2]), slope_sd = sd(co[,2]), intercept_mean = mean(co[,1]))
}

# A. Sequential
t_seq <- system.time({
  res_seq <- foreach(d = month_list, .combine = rbind) %do% boot_month(d)
})["elapsed"]
print(res_seq); cat("Sequential time:", t_seq, "s\n")

   month slope_mean    slope_sd intercept_mean
   <int>      <num>       <num>          <num>
1:     1   3.701873 0.006768768       6.171454
2:     2   3.717508 0.006238056       6.351019
3:     3   3.712085 0.006287445       6.510833
Sequential time: 10.876 s


In [ ]:
# B. Parallel (foreach + doParallel)
n_cores <- min(parallel::detectCores(), length(month_list))
cat("Using", n_cores, "cores\n")
cl <- makeCluster(n_cores)
registerDoParallel(cl)

t_par <- system.time({
  res_par <- foreach(d = month_list, .combine = rbind, .packages = "data.table") %dopar% boot_month(d)
})["elapsed"]
print(res_par); cat("Parallel time:", t_par, "s\n")

speedup <- t_seq / t_par
cat(sprintf("Speedup = %.2f / %.2f = %.2fx | Efficiency = %.0f%%\n", t_seq, t_par, speedup, 100 * speedup / n_cores))
stopCluster(cl); registerDoSEQ()

Using 2 cores
   month slope_mean    slope_sd intercept_mean
   <int>      <num>       <num>          <num>
1:     1   3.701873 0.006768768       6.171454
2:     2   3.717508 0.006238056       6.351019
3:     3   3.712085 0.006287445       6.510833
Parallel time: 12.453 s
Speedup = 10.88 / 12.45 = 0.87x | Efficiency = 44%


In [ ]:
# Extra: effect of number of workers and workload size
run_par <- function(k, B) {
  cl <- makeCluster(k); registerDoParallel(cl); on.exit({stopCluster(cl); registerDoSEQ()})
  system.time(foreach(d = month_list, .combine = rbind, .packages = "data.table") %dopar% boot_month(d, B = B))["elapsed"]
}
scal_par <- rbindlist(lapply(c(5, 25, 50), function(B) {
  ts <- system.time(foreach(d = month_list, .combine = rbind) %do% boot_month(d, B = B))["elapsed"]
  tp <- run_par(n_cores, B)
  data.table(B = B, Sequential_s = ts, Parallel_s = tp, Speedup = ts / tp)
}))
print(scal_par)

ERROR: Error in boot_month(d, B = B): task 1 failed - "could not find function "boot_month""


Timing stopped at: 0.399 0.054 0.889



### Interpretation (fill in after running)
* Speedup obtained: ___x with ___ cores (efficiency ___%).
* Colab typically gives only **2 vCPUs**, and there are only as many tasks as months, so the maximum possible speedup is limited by `min(cores, #partitions)`.
* Overheads: creating the PSOCK cluster, serializing the data to each worker, and combining results. For light workloads (small B), overhead can exceed the gain, so parallel may be **slower** (see the table above).
* Load imbalance: if one month is larger than the others, the whole run waits for it (Amdahl's law).

## 9. Task 5 (part 2): Comparison Table

In [ ]:
par_rows <- data.table(Task = "Bootstrap per month (parallel task)",
                       Approach = c("Sequential (foreach %do%)", paste0("Parallel (", n_cores, " cores)")),
                       Median_ms = c(t_seq, t_par) * 1000, Mean_ms = c(t_seq, t_par) * 1000, Max_ms = NA_real_)
full_rows <- data.table(Task = "Full-data aggregate", Approach = c("Base aggregate","data.table"),
                        Median_ms = c(t_base_full, t_dt_full) * 1000, Mean_ms = c(t_base_full, t_dt_full) * 1000, Max_ms = NA_real_)
comparison <- rbind(res_A, res_B, full_rows, par_rows)
comparison[, Rel_to_best := round(Median_ms / min(Median_ms), 1), by = Task]
print(comparison[, .(Task, Approach, Median_ms = round(Median_ms, 2), Rel_to_best)])
print(mem_tab)

## 10. Task 6: Data Visualization (ggplot2)

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 5.5)
th <- theme_minimal(base_size = 13) + theme(plot.title = element_text(face = "bold"))

ggplot(trips_hour, aes(hour, trips)) + geom_col(fill = "#f4b400") +
  scale_x_continuous(breaks = 0:23) + scale_y_continuous(labels = label_comma()) +
  labs(title = "Taxi trips by hour of day", subtitle = "Peak in the evening commute; trough in early morning",
       x = "Pickup hour (0-23)", y = "Number of trips") + th

In [ ]:
ggplot(trips_wday, aes(wday, trips, fill = wday %in% c("Sat","Sun"))) + geom_col() +
  scale_fill_manual(values = c("grey60","#db4437"), labels = c("Weekday","Weekend"), name = "Day type") +
  scale_y_continuous(labels = label_comma()) +
  labs(title = "Taxi demand by day of week", x = "Day of week", y = "Number of trips") + th

ggplot(trips_month, aes(factor(month), trips)) + geom_col(fill = "#4285f4") +
  scale_y_continuous(labels = label_comma()) +
  labs(title = "Monthly taxi demand", x = "Month (2024)", y = "Number of trips") + th

In [ ]:
ggplot(fare_hour, aes(hour, avg_fare)) + geom_line(linewidth = 1.2, colour = "#0f9d58") + geom_point() +
  scale_x_continuous(breaks = 0:23) +
  labs(title = "Average fare by hour", subtitle = "Early-morning trips are costlier (airport/long trips)",
       x = "Pickup hour", y = "Average fare (USD)") + th

ggplot(fare_hour, aes(hour, revenue / 1e6)) + geom_area(fill = "#ab47bc", alpha = .6) + geom_line() +
  scale_x_continuous(breaks = 0:23) +
  labs(title = "Total revenue by hour", x = "Pickup hour", y = "Revenue (million USD)") + th

In [ ]:
bar_top <- function(d, col, title, fill) {
  d <- copy(d); d[, lbl := reorder(get(col), trips)]
  ggplot(d, aes(lbl, trips)) + geom_col(fill = fill) + coord_flip() + scale_y_continuous(labels = label_comma()) +
    labs(title = title, x = NULL, y = "Number of trips") + th
}
bar_top(top_pu, "PU_Zone", "Top 15 pickup zones", "#f4b400")
bar_top(top_do, "DO_Zone", "Top 15 drop-off zones", "#4285f4")

In [ ]:
r1 <- copy(top_routes_freq)[, route := reorder(paste(PU_Zone, "→", DO_Zone), trips)]
ggplot(r1, aes(route, trips)) + geom_col(fill = "#db4437") + coord_flip() + scale_y_continuous(labels = label_comma()) +
  labs(title = "Top 15 most frequent routes", x = NULL, y = "Number of trips") + th

r2 <- copy(top_routes_rev)[, route := reorder(paste(PU_Zone, "→", DO_Zone), revenue)]
ggplot(r2, aes(route, revenue / 1e6)) + geom_col(fill = "#0f9d58") + coord_flip() +
  labs(title = "Top 15 highest-revenue routes", x = NULL, y = "Total revenue (million USD)") + th

In [ ]:
ggplot(dt[, .N, by = payment][, share := N / sum(N)], aes(reorder(payment, -N), share, fill = payment)) +
  geom_col(show.legend = FALSE) + geom_text(aes(label = percent(share, .1)), vjust = -.3) +
  scale_y_continuous(labels = percent) +
  labs(title = "Payment-type distribution", x = "Payment type", y = "Share of trips") + th

ggplot(pay_borough[PU_Borough %in% c("Manhattan","Brooklyn","Queens","Bronx","Staten Island","EWR")],
       aes(PU_Borough, share, fill = payment)) + geom_col() + scale_y_continuous(labels = percent) +
  labs(title = "Payment mix by pickup borough", x = "Pickup borough", y = "Share of trips", fill = "Payment") + th

In [ ]:
set.seed(1)
ggplot(dt[sample(.N, 2e5)], aes(trip_distance, fare_amount)) + geom_bin2d(bins = 80) +
  scale_fill_viridis_c(trans = "log10", name = "Trips (log)") + geom_smooth(method = "lm", colour = "red", se = FALSE) +
  coord_cartesian(xlim = c(0, 30), ylim = c(0, 120)) +
  labs(title = "Trip distance vs fare amount", subtitle = "200k-trip sample; red = linear fit",
       x = "Trip distance (miles)", y = "Fare amount (USD)") + th

In [ ]:
# Benchmark visualization
ggplot(comparison[grepl("^Op", Task)], aes(reorder(Approach, Median_ms), Median_ms, fill = Task)) +
  geom_col(show.legend = FALSE) + coord_flip() + scale_y_log10() + facet_wrap(~Task, scales = "free") +
  labs(title = "Execution time by approach (log scale)", x = NULL, y = "Median time (ms, log10)") + th

ggplot(melt(scal, id.vars = "n", variable.name = "Approach", value.name = "seconds"),
       aes(n, seconds, colour = Approach)) + geom_line(linewidth = 1) + geom_point() +
  scale_x_continuous(labels = label_comma()) +
  labs(title = "Scalability: time vs number of rows", x = "Rows", y = "Elapsed time (s)") + th

ggplot(data.table(Mode = c("Sequential", paste0("Parallel (", n_cores, " cores)")), Time = c(t_seq, t_par)),
       aes(Mode, Time, fill = Mode)) + geom_col(show.legend = FALSE) +
  labs(title = sprintf("Sequential vs parallel (speedup %.2fx)", speedup), x = NULL, y = "Elapsed time (s)") + th

**Optional (leaflet):** the zone lookup table has no coordinates; a map needs the TLC taxi-zone shapefile (`sf` package). Skip unless required.

**Visualization interpretation:** add 1–2 lines under each chart after running (e.g. demand peaks 18:00; Fridays busiest; Credit cards dominate; fare ≈ linear in distance).

In [ ]:
sessionInfo()